<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
This notebook adapts material from the <a href="https://docs.ros.org/en/jazzy/">ROS 2 documentation</a> (<a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>) and from the Duckietown <em>ROS Basics</em> and <em>ROS 2 Middleware</em> learning experiences.
</span>

# ROS 2 Discovery with zenoh, and Names

The previous notebook established that a publisher was running and that its messages reached a subscriber. Both terminals could discover the same node and refer to the same topic.

What happens if a publisher keeps running but another terminal cannot see it? How does a node started on your laptop find the nodes running inside Docker containers on the robot? And how can a topic appear under a name different from the one written in the publishing command?

This notebook investigates how ROS 2 participants discover one another in the Duckietown stack: through a **zenoh router**. A controlled change to the observer's router will make a running publisher disappear from its listing. A second experiment connects a terminal to the robot. A third one shows how ROS 2 resolves topic names. These observations help distinguish configuration and naming problems from problems in the application code.

## Discovery through middleware

A node listing is an observation of the graph visible to the observing process. **Discovery** is the process through which ROS 2 participants learn about one another and their interfaces. ROS 2 does not have a central "master" process for this: discovery is delegated to the middleware selected by the `RMW_IMPLEMENTATION` environment variable. Check it in a terminal:

```bash
printenv RMW_IMPLEMENTATION
```

The expected output in this editor, as on the robot, is:

```shell
rmw_zenoh_cpp
```

[zenoh](https://zenoh.io/) is a communication protocol with three kinds of participants:

- a **peer** talks directly to the other peers it can reach. Peers can find each other by multicast scouting on a local network, but that does not work through the Docker containers in which everything runs, nor across the WiFi between your laptop and the robot. The Duckietown stack therefore turns multicast off;
- a **router** is a process to which participants connect. It is a rendezvous point: it learns what everybody publishes and subscribes to, tells the participants about each other, and forwards data between those that cannot reach each other directly;
- a **client** does not talk to anybody directly: it connects to one router, and every message it sends or receives goes through that router.

On the Duckiebot the router runs in its own container, started by `dts stack up -H ROBOTNAME ros2/duckiebot`, and listens on `tcp/ROBOTNAME.local:7447`. Every ROS 2 node on the robot is a *peer* attached to that router. Anything that runs outside the robot (the terminals of this editor, the notebooks, the workbench container on your laptop) connects to the same router as a *client*, so that the robot routes everything: a peer session started on a laptop would try to link directly to the robot's containers, which are unreachable from outside, and would see nothing. The configuration is one environment variable:

```bash
export ZENOH_CONFIG_OVERRIDE='mode="client";connect/endpoints=["tcp/ROBOTNAME.local:7447"]'
```

Every Duckietown container derives this variable from `VEHICLE_NAME`: on the robot and in the workbench container (`dts code workbench -R ROBOTNAME`) that is the robot's name, and the `RosSession("ROBOTNAME")` cell of the notebooks does the same for the notebook kernel. The terminals of this editor also derive it, but the editor has no robot of its own: its `VEHICLE_NAME` is the container's own hostname, so every terminal starts as a client of `tcp/<hostname>.local:7447`, i.e. of a router *inside the editor*. That is the router you start with `rmw_zenohd` in notebook 03, and that is why that command is prefixed with `ZENOH_CONFIG_OVERRIDE=`: the variable applies to every zenoh process, and a router started with it would try to become a client itself. To work with a robot instead, a terminal exports the variable with the robot's address, as above.

One consequence for the command-line tools: every `ros2` command is a new session that connects to the router and then needs a few seconds to learn the graph. A listing taken too early is empty. `ros2 node list --no-daemon --spin-time 5` makes the tool wait five seconds before answering, which is why this form is used throughout these notebooks whenever the graph is inspected.

## Experiment 1: what happens without a router

As in the previous notebook, Terminal 1 runs the publisher and Terminal 2 observes it. Make sure a router is reachable (the robot's, or the local `rmw_zenohd` of notebook 03).

In Terminal 1, start the familiar text publisher. This time the topic argument is written as `chatter`, without the initial slash; its resolved name will be inspected below.

```bash
ros2 topic pub --rate 1 --node-name example_publisher chatter std_msgs/msg/String "{data: '42'}"
```

In Terminal 2, observe directly (bypassing the background helper of the command line tools, the *daemon*, which keeps its own cached view of the graph) and allow five seconds for discovery:

```bash
ros2 node list --no-daemon --spin-time 5
```

The expected output is:

```shell
/example_publisher
```

(plus the robot's nodes if you are connected to a robot). The observer has discovered the node running in Terminal 1.

Now repeat the observation while pointing this single command at a router that does not exist, port 7448 of the local machine:

```bash
ZENOH_CONFIG_OVERRIDE='mode="client";connect/endpoints=["tcp/127.0.0.1:7448"]' ros2 node list --no-daemon --spin-time 5
```

The command warns, waits and prints no node:

```shell
[WARN] [rmw_zenoh_cpp]: Unable to connect to a Zenoh router. Have you started a router with `ros2 run rmw_zenoh_cpp rmw_zenohd`?
[WARN] [rmw_zenoh_cpp]: Unable to connect to a Zenoh router after 1 attempt(s). ... Proceeding with initialization but other peers will not discover or receive data from peers in this session until a router is started.
```

The publisher remained active throughout. Changing the observer's router changed whether it could discover that publisher. The variable assignment placed before the command applies only to that command; it does not change Terminal 2's environment or the running publisher.

An empty node listing therefore means **no nodes were discovered by this observation**. It does not establish that no nodes are running. Whenever a listing is unexpectedly empty, check three things, in this order:

1. is the publisher process still running?
2. do the observer and the publisher connect to the same router (`printenv ZENOH_CONFIG_OVERRIDE` in both terminals)?
3. did discovery have enough time (`--no-daemon --spin-time 5`)?

The daemon deserves one more remark: `ros2 node list` without `--no-daemon` asks a background process that was started by the first `ros2` command of the session, with that command's router. If you change `ZENOH_CONFIG_OVERRIDE` in a terminal, run `ros2 daemon stop` once so that the next listing starts a fresh daemon with the new configuration.

## Experiment 2: connect a terminal to the robot

This experiment needs a real or virtual Duckiebot with its ROS 2 stack running (see the README: `dts stack up -H ROBOTNAME ros2/duckiebot` and `dts stack up -H ROBOTNAME ros2-core/default`). Skip it if you do not have one yet; notebook 08 repeats it in depth.

Open a new terminal and point it at the robot's router:

```bash
export ZENOH_CONFIG_OVERRIDE='mode="client";connect/endpoints=["tcp/ROBOTNAME.local:7447"]'
ros2 daemon stop
```

(`ros2 daemon stop` discards the cached graph of the editor's local router; the next command starts a fresh daemon with the new configuration.) Check the configuration with `printenv ZENOH_CONFIG_OVERRIDE`, then:

```bash
ros2 node list --no-daemon --spin-time 5
```

The listing now contains the robot's nodes, in the robot's namespace:

```shell
/ROBOTNAME/camera_node
/ROBOTNAME/car_cmd_switch_node
/ROBOTNAME/kinematics_node
/ROBOTNAME/led_driver_node
/ROBOTNAME/left_wheel_encoder_driver_node
/ROBOTNAME/right_wheel_encoder_driver_node
/ROBOTNAME/tof_driver_node
/ROBOTNAME/velocity_to_pose_node
/ROBOTNAME/wheels_driver_node
```

(names may be fewer or more depending on which stacks are up). Your `example_publisher` is not in this list if it was started with a different router: the two graphs are separate. Stop it and start it again from a terminal configured for the robot, and it joins the robot's graph; the robot's nodes could subscribe to it.

The same `ros2 topic echo --once /ROBOTNAME/left_wheel_encoder_driver_node/tick` now receives a message from the robot: the router makes the location of a node irrelevant to the tools. Your own nodes started from these terminals (notebook 09) are clients of the same router, and the robot's nodes see their topics and services exactly as if they ran on the robot.

## Absolute, relative and private names

Discovery makes participants visible, but a subscriber must then still refer to the intended topic by name. The publishing command of experiment 1 used `chatter` as topic name, but the graph displays resolved names. With the publisher running, in Terminal 2:

```bash
ros2 topic list -t --no-daemon --spin-time 5
```

The expected output includes:

```shell
/chatter [std_msgs/msg/String]
```

The topic argument `chatter` became `/chatter`.

A **namespace** supplies a prefix for names. This publisher runs in the root (`/`) namespace, so the relative name `chatter` resolves to the absolute `/chatter`. An absolute name already starts with `/` and specifies the complete name.

### Make a topic follow its node's name

Suppose each instance of a component needs a separate topic for reporting its own state. A **private name**, beginning with `~/`, includes the node's namespace and name. "Private" describes how the name is formed; it does not restrict access to the topic.

Stop the publisher with `Ctrl-C` in Terminal 1, then start it again with:

```bash
ros2 topic pub --rate 1 --node-name example_publisher '~/state' std_msgs/msg/String "{data: '42'}"
```

The quotes preserve `~/state` for ROS 2 instead of letting the shell interpret the tilde as your home directory. Inspect the topics in Terminal 2:

```bash
ros2 topic list -t --no-daemon --spin-time 5
```

The listing should include:

```shell
/example_publisher/state [std_msgs/msg/String]
```

ROS 2 expanded `~` using the publishing node's name. The topic remains accessible through its resolved name:

```bash
ros2 topic echo --once /example_publisher/state
```

### Predict the effect of a different node name

If the topic argument remains `~/state` but the node becomes `second_publisher`, which topic should appear? Stop the publisher in Terminal 1, then start:

```bash
ros2 topic pub --rate 1 --node-name second_publisher '~/state' std_msgs/msg/String "{data: '42'}"
```

In Terminal 2 the relevant entry of `ros2 topic list -t --no-daemon --spin-time 5` should now be:

```shell
/second_publisher/state [std_msgs/msg/String]
```

Changing the node name changed the resolved private topic. Distinct node names give separate instances distinct private topics.

### The robot's namespace

This is exactly how the Duckiebot's topics are named. Every node of the robot runs in the namespace `/ROBOTNAME`, and the drivers publish on private names: `camera_node` publishes `~/image/compressed`, which resolves to `/ROBOTNAME/camera_node/image/compressed`. Table 1 applies the resolution rules to the node `/ROBOTNAME/camera_node`.

| Form | Topic argument | Resolved topic |
| --- | --- | --- |
| Absolute | `/image` | `/image` |
| Relative | `image` | `/ROBOTNAME/image` |
| Private | `~/image` | `/ROBOTNAME/camera_node/image` |

*Table 1: Topic name resolution for the node `/ROBOTNAME/camera_node`.*

The namespace is not written in the code of the nodes: it is given when they are started (a launch file argument, as you will see in notebook 09). The same code therefore works on any robot, and two robots on the same network do not mix their topics. Your own nodes in this learning experience follow the rule: they use relative names such as `camera_node/image/compressed`, and their launch file puts them in the robot's namespace.

## Finish the observation

Stop the publisher in Terminal 1 with `Ctrl-C`. In Terminal 2:

```bash
ros2 node list --no-daemon --spin-time 5
```

The listing should become empty again (apart from the robot's nodes) after the publisher stops.

A running publisher disappeared from an observer's listing when their routers differed, then reappeared when they matched; connecting a terminal to the robot's router made the robot's nodes visible. A private topic changed its resolved name when its publishing node was renamed, and the robot's namespace explained the names seen on the Duckiebot. These observations are the first things to check when a message does not arrive.

The command-line publisher supplied the application behaviour for these experiments. Implementing a custom node also requires organising its source and making it available to ROS 2, which is the subject of the next notebook.

## Further reading

The [rmw_zenoh documentation](https://github.com/ros2/rmw_zenoh) explains the router, the session configuration and the `ZENOH_CONFIG_OVERRIDE` variable; the [zenoh documentation](https://zenoh.io/docs/getting-started/deployment/) describes peers, clients and routers. The ROS 2 design article covers [topic and service names](https://design.ros2.org/articles/topic_and_service_names.html).

Let's move to [the next notebook about workspaces and packages](../notebooks/05_workspaces_and_packages.ipynb).